In [1]:
import pandas as pd
import numpy as np

In [2]:
matches = pd.read_csv("../Dataset/matches.csv")
deliveries = pd.read_csv("../Dataset/deliveries.csv")

In [3]:
print(matches.shape)
print(deliveries.shape)
print(matches.isna().sum())
print(deliveries.isna().sum())

(1095, 20)
(260920, 17)
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [4]:
# ============================================================
# STEP 1 — BASIC MATCH CLEANING + MERGING
# ============================================================

# Convert date
matches["date"] = pd.to_datetime(
    matches["date"],
    errors="coerce"
)

# Remove matches without a winner
matches = matches.dropna(
    subset=["winner"]
).copy()

# Fill missing cities
matches["city"] = matches["city"].fillna("Unknown")


# ------------------------------------------------------------
# Normalize team names
# ------------------------------------------------------------

team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant"
}

matches["team1"] = matches["team1"].replace(team_mapping)
matches["team2"] = matches["team2"].replace(team_mapping)
matches["winner"] = matches["winner"].replace(team_mapping)

deliveries["batting_team"] = (
    deliveries["batting_team"].replace(team_mapping)
)

deliveries["bowling_team"] = (
    deliveries["bowling_team"].replace(team_mapping)
)


# ------------------------------------------------------------
# Normalize city
# ------------------------------------------------------------

city_mapping = {
    "Bangalore": "Bengaluru"
}

matches["city"] = matches["city"].replace(city_mapping)


# ------------------------------------------------------------
# Merge match-level information
# ------------------------------------------------------------

df = deliveries.merge(
    matches[
        [
            "id",
            "season",
            "city",
            "date",
            "venue",
            "toss_winner",
            "toss_decision",
            "winner",
            "target_runs",
            "target_overs",
            "method"
        ]
    ],
    left_on="match_id",
    right_on="id",
    how="inner"
)

print("Merged shape:", df.shape)

Merged shape: (260430, 28)


In [5]:
df.sample(10)

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,season,city,date,venue,toss_winner,toss_decision,winner,target_runs,target_overs,method
72143,548364,2,Chennai Super Kings,Delhi Capitals,9,4,M Vijay,RE van der Merwe,MEK Hussey,0,...,2012,Chennai,2012-05-12,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,field,Chennai Super Kings,115.0,20.0,NaN
133028,980993,1,Mumbai Indians,Delhi Capitals,2,1,RG Sharma,Z Khan,MJ Guptill,0,...,2016,Visakhapatnam,2016-05-15,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,Delhi Daredevils,field,Mumbai Indians,207.0,20.0,NaN
145230,1082629,2,Rising Pune Supergiant,Gujarat Lions,12,5,MS Dhoni,Ankit Soni,BA Stokes,0,...,2017,Pune,2017-05-01,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,field,Rising Pune Supergiant,162.0,20.0,NaN
12227,336035,2,Kolkata Knight Riders,Punjab Kings,8,2,DJ Hussey,PP Chawla,SC Ganguly,0,...,2007/08,Kolkata,2008-05-25,Eden Gardens,Kings XI Punjab,bat,Kolkata Knight Riders,175.0,20.0,NaN
250941,1426270,1,Gujarat Titans,Delhi Capitals,0,5,WP Saha,KK Ahmed,Shubman Gill,0,...,2024,Ahmedabad,2024-04-17,"Narendra Modi Stadium, Ahmedabad",Delhi Capitals,field,Delhi Capitals,90.0,20.0,NaN
67975,548347,2,Kolkata Knight Riders,Chennai Super Kings,1,7,G Gambhir,JA Morkel,BB McCullum,0,...,2012,Chennai,2012-04-30,"MA Chidambaram Stadium, Chepauk",Chennai Super Kings,bat,Kolkata Knight Riders,140.0,20.0,NaN
175619,1178422,1,Kolkata Knight Riders,Mumbai Indians,8,4,CA Lynn,BB Sran,Shubman Gill,1,...,2019,Kolkata,2019-04-28,Eden Gardens,Mumbai Indians,field,Kolkata Knight Riders,233.0,20.0,NaN
148800,1082644,2,Kolkata Knight Riders,Mumbai Indians,11,1,MK Pandey,KH Pandya,C de Grandhomme,1,...,2017,Kolkata,2017-05-13,Eden Gardens,Kolkata Knight Riders,field,Mumbai Indians,174.0,20.0,NaN
10203,336026,1,Royal Challengers Bengaluru,Delhi Capitals,14,3,SP Goswami,VY Mahesh,Misbah-ul-Haq,4,...,2007/08,Bengaluru,2008-05-19,M Chinnaswamy Stadium,Delhi Daredevils,field,Delhi Capitals,155.0,20.0,NaN
80643,598015,2,Chennai Super Kings,Pune Warriors,18,2,JA Morkel,B Kumar,CH Morris,0,...,2013,Chennai,2013-04-15,"MA Chidambaram Stadium, Chepauk",Pune Warriors,bat,Pune Warriors,160.0,20.0,NaN


In [6]:
# ============================================================
# STEP 2 — CREATE LIVE MATCH STATE DATASET
# ============================================================

# ------------------------------------------------------------
# Keep only second innings
# ------------------------------------------------------------

df = df[df["inning"] == 2].copy()


# ------------------------------------------------------------
# Current score
# ------------------------------------------------------------

df["current_score"] = (
    df.groupby("match_id")["total_runs"]
      .cumsum()
)


# ------------------------------------------------------------
# RUNS LEFT
#
# Use target_runs instead of first innings total.
# This correctly handles D/L matches.
# ------------------------------------------------------------

df["runs_left"] = (
    df["target_runs"] - df["current_score"]
)


# ------------------------------------------------------------
# LEGAL BALLS BOWLED
#
# A wide/no-ball does NOT consume a legal ball.
# ------------------------------------------------------------

df["legal_ball"] = (
    ~df["extras_type"].isin(
        ["wides", "noballs"]
    )
).astype(int)


df["legal_balls_bowled"] = (
    df.groupby("match_id")["legal_ball"]
      .cumsum()
)


# ------------------------------------------------------------
# BALLS LEFT
# ------------------------------------------------------------

df["balls_left"] = (
    (df["target_overs"] * 6)
    - df["legal_balls_bowled"]
)


# ------------------------------------------------------------
# WICKETS LEFT
# ------------------------------------------------------------

df["wickets_lost"] = (
    df.groupby("match_id")["is_wicket"]
      .cumsum()
)

df["wickets_left"] = (
    10 - df["wickets_lost"]
)


# ------------------------------------------------------------
# CURRENT RUN RATE
# ------------------------------------------------------------

df["crr"] = np.where(
    df["legal_balls_bowled"] > 0,

    df["current_score"] * 6
    / df["legal_balls_bowled"],

    0
)


# ------------------------------------------------------------
# REQUIRED RUN RATE
# ------------------------------------------------------------

df["rrr"] = np.where(
    df["balls_left"] > 0,

    df["runs_left"] * 6
    / df["balls_left"],

    0
)


# ------------------------------------------------------------
# RESULT
#
# 1 = batting team eventually won
# 0 = batting team eventually lost
# ------------------------------------------------------------

df["result"] = (
    df["batting_team"] == df["winner"]
).astype(int)


# ------------------------------------------------------------
# Select final features
# ------------------------------------------------------------

final_df = df[
    [
        "match_id",
        "season",
        "date",

        "batting_team",
        "bowling_team",
        "city",

        "runs_left",
        "balls_left",
        "wickets_left",

        "target_runs",

        "crr",
        "rrr",

        "result"
    ]
].copy()


print("Before final cleaning:", final_df.shape)

Before final cleaning: (125714, 13)


In [7]:
# ============================================================
# STEP 3 — FINAL CLEANING
# ============================================================

# ------------------------------------------------------------
# Remove impossible / terminal states
# ------------------------------------------------------------

final_df = final_df[
    (final_df["runs_left"] > 0) &
    (final_df["balls_left"] > 0) &
    (final_df["wickets_left"] > 0)
].copy()


# ------------------------------------------------------------
# Remove infinite values
# ------------------------------------------------------------

final_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


# ------------------------------------------------------------
# Remove missing values
# ------------------------------------------------------------

required_columns = [
    "match_id",
    "season",
    "date",
    "batting_team",
    "bowling_team",
    "city",
    "runs_left",
    "balls_left",
    "wickets_left",
    "target_runs",
    "crr",
    "rrr",
    "result"
]

final_df.dropna(
    subset=required_columns,
    inplace=True
)


# ------------------------------------------------------------
# Convert numerical columns
# ------------------------------------------------------------

numeric_columns = [
    "runs_left",
    "balls_left",
    "wickets_left",
    "target_runs",
    "crr",
    "rrr",
    "result"
]

for col in numeric_columns:

    final_df[col] = pd.to_numeric(
        final_df[col],
        errors="coerce"
    )


final_df["result"] = (
    final_df["result"].astype(int)
)


# ------------------------------------------------------------
# Convert season to numeric year
# ------------------------------------------------------------

def get_season_year(value):

    value = str(value)

    # Special IPL season labels
    if value == "2007/08":
        return 2008

    if value == "2009/10":
        return 2010

    if value == "2020/21":
        return 2020

    return int(value)


final_df["season_year"] = (
    final_df["season"]
    .apply(get_season_year)
)


# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

final_df = final_df.sort_values(
    [
        "season_year",
        "date",
        "match_id"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# Final column order
# ------------------------------------------------------------

final_df = final_df[
    [
        "match_id",
        "season_year",
        "date",

        "batting_team",
        "bowling_team",
        "city",

        "runs_left",
        "balls_left",
        "wickets_left",

        "target_runs",

        "crr",
        "rrr",

        "result"
    ]
]


# ============================================================
# FINAL CHECK
# ============================================================

print("\n========== FINAL DATASET ==========")

print(
    "Shape:",
    final_df.shape
)

print("\nMissing values:")
print(
    final_df.isna().sum()
)

print("\nTeams:")

print(
    sorted(
        set(final_df["batting_team"])
        |
        set(final_df["bowling_team"])
    )
)

print("\nSeasons:")

print(
    sorted(
        final_df["season_year"].unique()
    )
)

print("\nTarget distribution:")

print(
    final_df["result"].value_counts()
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

final_df.to_csv(
    "../Dataset/ipl_match_state_clean.csv",
    index=False
)

print(
    "\nSaved successfully:"
)

print(
    "../Dataset/ipl_match_state_clean.csv"
)


========== FINAL DATASET ==========
Shape: (124563, 13)

Missing values:
match_id        0
season_year     0
date            0
batting_team    0
bowling_team    0
city            0
runs_left       0
balls_left      0
wickets_left    0
target_runs     0
crr             0
rrr             0
result          0
dtype: int64

Teams:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Seasons:
[np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Target distribution:
result
1    64696
0    59867
Name: count, dtyp

In [10]:
# ============================================================
# STEP 04 — LOGISTIC REGRESSION MODEL
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("../Dataset/ipl_match_state.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. NORMALIZE TEAM NAMES
# ------------------------------------------------------------

team_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant"
}

df["batting_team"] = df["batting_team"].replace(team_mapping)
df["bowling_team"] = df["bowling_team"].replace(team_mapping)

# City normalization
df["city"] = df["city"].replace({
    "Bangalore": "Bengaluru"
})


# ------------------------------------------------------------
# 3. CONVERT SEASON TO NUMERIC YEAR
# ------------------------------------------------------------

def season_to_year(value):

    value = str(value)

    if "/" in value:
        parts = value.split("/")
        return int(parts[-1])

    return int(value)


df["season_year"] = df["season"].apply(season_to_year)


# ------------------------------------------------------------
# 4. FEATURES + TARGET
# ------------------------------------------------------------

features = [
    "batting_team",
    "bowling_team",
    "city",
    "runs_left",
    "balls_left",
    "wickets_left",
    "total_runs_x",
    "crr",
    "rrr"
]

target = "result"

X = df[features].copy()
y = df[target].copy()


# ------------------------------------------------------------
# 5. REMOVE INVALID VALUES
# ------------------------------------------------------------

X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = X.notna().all(axis=1)

X = X[valid_rows]
y = y[valid_rows]
df_clean = df.loc[X.index].copy()


# ------------------------------------------------------------
# 6. CHRONOLOGICAL TRAIN / TEST SPLIT
# ------------------------------------------------------------
# Train: 2008–2023
# Test : 2024
#
# This avoids random row leakage because multiple rows
# belong to the same IPL match.

train_mask = df_clean["season_year"] <= 2023
test_mask = df_clean["season_year"] == 2024

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTrain seasons:")
print(sorted(df_clean.loc[train_mask, "season_year"].unique()))

print("\nTest seasons:")
print(sorted(df_clean.loc[test_mask, "season_year"].unique()))


# ------------------------------------------------------------
# 7. DEFINE CATEGORICAL + NUMERICAL FEATURES
# ------------------------------------------------------------

categorical_features = [
    "batting_team",
    "bowling_team",
    "city"
]

numerical_features = [
    "runs_left",
    "balls_left",
    "wickets_left",
    "total_runs_x",
    "crr",
    "rrr"
]


# ------------------------------------------------------------
# 8. PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),

        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)


# ------------------------------------------------------------
# 9. LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="liblinear",
    random_state=42
)


# ------------------------------------------------------------
# 10. COMPLETE PIPELINE
# ------------------------------------------------------------

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logistic_model)
    ]
)


# ------------------------------------------------------------
# 11. TRAIN
# ------------------------------------------------------------

print("\nTraining Logistic Regression...")

logistic_pipeline.fit(
    X_train,
    y_train
)

print("Training completed.")


# ------------------------------------------------------------
# 12. PREDICTION
# ------------------------------------------------------------

y_pred = logistic_pipeline.predict(X_test)

y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]


# ------------------------------------------------------------
# 13. EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

logloss = log_loss(
    y_test,
    y_prob
)

brier = brier_score_loss(
    y_test,
    y_prob
)


print("\n" + "=" * 60)
print("LOGISTIC REGRESSION — 2024 TEST RESULTS")
print("=" * 60)

print(f"Accuracy   : {accuracy:.4f}")
print(f"Log Loss   : {logloss:.4f}")
print(f"Brier Score: {brier:.4f}")


# ------------------------------------------------------------
# 14. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Bowling Team Wins",
            "Batting Team Wins"
        ]
    )
)


# ------------------------------------------------------------
# 15. CONFUSION MATRIX
# ------------------------------------------------------------

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)


# ------------------------------------------------------------
# 16. TRAIN FINAL MODEL ON ALL AVAILABLE DATA
# ------------------------------------------------------------
# After evaluating on untouched 2024 data,
# train the deployment model using ALL available data.

print("\nTraining final deployment model...")

final_model = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "categorical",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        ),
                        categorical_features
                    ),
                    (
                        "numerical",
                        StandardScaler(),
                        numerical_features
                    )
                ]
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                C=1.0,
                solver="liblinear",
                random_state=42
            )
        )
    ]
)


final_model.fit(
    X,
    y
)

print("Final model trained.")


# ------------------------------------------------------------
# 17. SAVE MODEL
# ------------------------------------------------------------

joblib.dump(
    final_model,
    "../models/logistic_model.pkl"
)

print("\nModel saved successfully:")
print("logistic_model.pkl")


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)

print("Model       : Logistic Regression")
print("Features    :", len(features))
print("Train/Test  : <= 2023 / 2024")
print(f"Accuracy    : {accuracy:.2%}")
print(f"Log Loss    : {logloss:.4f}")
print(f"Brier Score : {brier:.4f}")
print("=" * 60)

Dataset shape: (125010, 13)

Columns:
['match_id', 'season', 'date', 'batting_team', 'bowling_team', 'city', 'runs_left', 'balls_left', 'wickets_left', 'total_runs_x', 'crr', 'rrr', 'result']

Training samples: 116845
Testing samples : 8165

Train seasons:
[np.int64(8), np.int64(10), np.int64(21), np.int64(2009), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023)]

Test seasons:
[np.int64(2024)]

Training Logistic Regression...
Training completed.

LOGISTIC REGRESSION — 2024 TEST RESULTS
Accuracy   : 0.7858
Log Loss   : 0.5034
Brier Score: 0.1584

Classification Report:
                   precision    recall  f1-score   support

Bowling Team Wins       0.75      0.89      0.81      4281
Batting Team Wins       0.85      0.67      0.75      3884

         accuracy                           0.79      8165
        macro avg       0.80      0.78      0